# 🤖 HyperFlow H3 Flash - Autonomous Batch Runner (Headless Mode)
### Automated Headless Ref2VA Joint Video & Audio Rendering for Antigravity Agents
* **Mode:** **Autonomous Headless Batch**
* **Behavior:**
  1. Executes batch jobs defined in Cell 1 sequentially.
  2. Each job is isolated in a `try...except` block (errors never halt the batch).
  3. Writes rendered outputs immediately to `/kaggle/working/outputs/`.
  4. Automatically terminates the kernel (`sys.exit(0)`) on completion to **save GPU quota**.


In [ ]:
# [CONFIGURATION] Antigravity Autonomous Batch Job Definitions
# Modify or inject this job list programmatically before dispatching.

BATCH_JOBS = [
    {
        "id": "shot_001_walkthrough",
        "prompt": "<Picture 1> walks slowly through the scene of <Picture 2>, looks around and smiles; the camera follows at shoulder height, footsteps and soft ambient sound fill the space.",
        "ref_images": ["ref_image_1.png", "ref_image_2.png"],
        "seed": 777,
        "steps": 25,
        "fps": 24
    },
    {
        "id": "shot_002_handheld_vlog",
        "prompt": "<Picture 1> holds a smartphone camera in front of him for an energetic daily vlog on a sunlit urban street, talking directly into the lens with natural hand gestures and organic handheld camera micro-shake: <d>[English] What is up everyone! Today we are testing this brand new joint video and audio generation model for the first time, and the lip sync is unbelievable.</d> City traffic ambiance, soft passing pedestrians, crisp outdoor vocal acoustics.",
        "ref_images": ["vlog_creator.png"],
        "seed": 42,
        "steps": 25,
        "fps": 24
    },
    {
        "id": "shot_003_cinematic_landscape",
        "prompt": "<Picture 1> A breathtaking cinematic extreme wide shot slowly sweeping over an ancient misty temple nestled in towering pine-covered mountain peaks at sunrise. Golden morning light cuts through drifting fog layers as distant birds soar across the valley. Gentle mountain breeze rustles the trees, distant temple bell resonance, deep atmospheric ambient soundscape.",
        "ref_images": ["wide_temple.png"],
        "seed": 101,
        "steps": 25,
        "fps": 24
    },
    {
        "id": "shot_004_two_person_dialogue",
        "prompt": "subject_definitions: <Subject 1> is the detective in <Picture 1>. <Subject 2> is the informant in <Picture 2>. detailed_description: In a dim rainy neon alleyway, <Subject 1> steps forward and speaks sternly to <Subject 2>: <d>[English] You told me this drop was clean, so why are the feds already on the block?</d> <Subject 2> does not speak and his lips remain completely closed, nervously glancing sideways. overall_soundscape: Torrential rain splashing on asphalt, distant police siren echoing through the city.",
        "ref_images": ["char_detective.png", "char_informant.png"],
        "seed": 202,
        "steps": 25,
        "fps": 24
    },
    {
        "id": "shot_005_martial_arts_action",
        "prompt": "Dynamic high-speed martial arts combat in an industrial warehouse between <Picture 1> and <Picture 2>. Fighter 1 delivers a rapid high roundhouse kick which Fighter 2 ducks under and retaliates with a swift double palm strike. The camera vigorously tracks and whip-pans with the action. Fast whooshes of punches cutting through the air, heavy physical impact thuds, squeaking combat boots on concrete, intense exertion grunts: <d>[English] Hah! ... Oof!</d>",
        "ref_images": ["fighter_red.png", "fighter_black.png"],
        "seed": 303,
        "steps": 25,
        "fps": 24
    }
]

print(f"Loaded {len(BATCH_JOBS)} autonomous batch job(s) for headless execution.", flush=True)


In [ ]:
# [Step 1/4] Environment & ComfyUI Setup
import os, sys, subprocess, time

try:
    sys.stdout.reconfigure(line_buffering=True)
    sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass

print(">>> [1/4] Setting up ComfyUI Core...", flush=True)
if not os.path.exists("ComfyUI"):
    subprocess.run("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git", shell=True, check=True)

custom_nodes_path = "ComfyUI/custom_nodes/ComfyUI-HyperFlow-Flash"
if not os.path.exists(custom_nodes_path):
    subprocess.run("git clone --depth 1 https://github.com/Video-Rebirth/ComfyUI-HyperFlow-Flash.git " + custom_nodes_path, shell=True, check=True)

pkgs = ["huggingface_hub>=0.28.0", "hf_transfer", "gguf", "pillow", "requests", "tqdm"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "ComfyUI/requirements.txt"], check=True)
print("✓ Environment ready.", flush=True)


In [ ]:
# [Step 2/4] High-Speed Model Ingestion (Mounted Dataset + HF Fallback)
import os, shutil, glob, time
from huggingface_hub import hf_hub_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.makedirs("ComfyUI/models/diffusion_models", exist_ok=True)
os.makedirs("ComfyUI/models/text_encoders", exist_ok=True)
os.makedirs("ComfyUI/models/vae", exist_ok=True)

ds_candidates = glob.glob('/kaggle/input/*minimax*weights*')
ds_root = ds_candidates[0] if ds_candidates else '/kaggle/input/minimax-h3-open-weights'

def link_or_fetch(rel_ds_path, local_dst, hf_repo, hf_file):
    src = os.path.join(ds_root, rel_ds_path)
    if os.path.exists(src):
        if not os.path.exists(local_dst):
            try:
                os.symlink(src, local_dst)
            except Exception:
                shutil.copyfile(src, local_dst)
        print(f"✓ Linked from mounted dataset: {local_dst}", flush=True)
    else:
        if not os.path.exists(local_dst):
            print(f"Downloading from Hugging Face: {hf_file}...", flush=True)
            p = hf_hub_download(repo_id=hf_repo, filename=hf_file)
            try:
                os.symlink(p, local_dst)
            except Exception:
                shutil.copyfile(p, local_dst)
            print(f"✓ Downloaded: {local_dst}", flush=True)

link_or_fetch(
    "clip/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    "ComfyUI/models/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    "Comfy-Org/MiniMax-H3",
    "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
)
link_or_fetch(
    "vae/minimax_h3_video_vae_int8_convrot.safetensors",
    "ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors",
    "Comfy-Org/MiniMax-H3",
    "vae/minimax_h3_video_vae_fp16.safetensors"
)
link_or_fetch(
    "vae/minimax_h3_audio_vae_fp32.safetensors",
    "ComfyUI/models/vae/minimax_h3_audio_vae_fp32.safetensors",
    "Comfy-Org/MiniMax-H3",
    "vae/minimax_h3_audio_vae_fp32.safetensors"
)
link_or_fetch(
    "diffusion_models/minimax_h3_ref2va_hyperflow_flash_14b-int8_convrot.safetensors",
    "ComfyUI/models/diffusion_models/minimax_h3_ref2va_hyperflow_flash_14b-int8_convrot.safetensors",
    "VideoRebirthAI/hyperflow-h3-flash",
    "ref2va_int8_convrot_comfyui/minimax_h3_ref2va_hyperflow_flash_14b-int8_convrot.safetensors"
)
print("✓ Checkpoints mounted and verified.", flush=True)


In [ ]:
# [Step 3/4] Setup Reference Input Assets & Synthetic Placeholders
import os, urllib.request
from PIL import Image, ImageDraw

input_dir = "ComfyUI/input"
os.makedirs(input_dir, exist_ok=True)

official_urls = {
    "ref_image_1.png": "https://raw.githubusercontent.com/Video-Rebirth/hyperflow-h3-flash/main/integrations/ComfyUI-HyperFlow-Flash/example_workflows/hyperflow_ref2v_int8.jpg",
    "ref_image_2.png": "https://raw.githubusercontent.com/Video-Rebirth/hyperflow-h3-flash/main/integrations/ComfyUI-HyperFlow-Flash/example_workflows/hyperflow_i2v_int8.jpg"
}
for fname, url in official_urls.items():
    dst = os.path.join(input_dir, fname)
    if not os.path.exists(dst):
        try:
            urllib.request.urlretrieve(url, dst)
        except Exception:
            pass

def create_reference_canvas(filename, bg_color, accent_color, title_text):
    path = os.path.join(input_dir, filename)
    if os.path.exists(path):
        return
    img = Image.new("RGB", (768, 768), color=bg_color)
    draw = ImageDraw.Draw(img)
    draw.ellipse([264, 184, 504, 424], fill=accent_color)
    draw.rectangle([20, 20, 748, 80], fill=(20, 20, 20))
    draw.text((40, 40), f"REF: {title_text}", fill=(255, 255, 255))
    img.save(path)

create_reference_canvas("vlog_creator.png", (40, 50, 65), (70, 90, 120), "VLOG CREATOR")
create_reference_canvas("wide_temple.png", (25, 45, 30), (50, 80, 60), "MOUNTAIN TEMPLE")
create_reference_canvas("char_detective.png", (30, 30, 45), (60, 60, 80), "DETECTIVE")
create_reference_canvas("char_informant.png", (45, 30, 30), (90, 60, 60), "INFORMANT")
create_reference_canvas("fighter_red.png", (60, 20, 20), (120, 40, 40), "FIGHTER RED")
create_reference_canvas("fighter_black.png", (20, 20, 25), (45, 45, 55), "FIGHTER BLACK")
print("✓ Input references provisioned.", flush=True)


In [ ]:
# [Step 4/4] Headless Server Launch, Fault-Tolerant Batch Execution & Auto-Termination
import subprocess, threading, time, re, urllib.request, json, shutil

PORT = 8188
OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 75, flush=True)
print(">>> Starting Headless ComfyUI Server...", flush=True)
print("=" * 75, flush=True)

comfy_proc = subprocess.Popen(
    f"{sys.executable} ComfyUI/main.py --listen 127.0.0.1 --port {PORT} --enable-cors-header --preview-method none",
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

def monitor():
    for line in comfy_proc.stdout:
        if any(w in line for w in ["Executing", "HyperFlow", "error", "Error", "CUDA"]):
            print(f"[ComfyUI] {line.strip()}", flush=True)

threading.Thread(target=monitor, daemon=True).start()

# Wait for local API readiness
server_ready = False
for _ in range(60):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/system_stats", timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                print("✓ Headless ComfyUI API is online!", flush=True)
                break
    except Exception:
        time.sleep(2)

if not server_ready:
    raise RuntimeError("ComfyUI server failed to start within 120s.")

def build_workflow(prompt_text, ref_images, output_prefix, seed=42):
    return {
        "1": {"class_type": "HyperFlowLoader", "inputs": {"model_name": "minimax_h3_ref2va_hyperflow_flash_14b-int8_convrot.safetensors", "text_encoder": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "video_vae": "minimax_h3_video_vae_fp16.safetensors", "audio_vae": "minimax_h3_audio_vae_fp32.safetensors", "precision": "int8_convrot"}},
        "2": {"class_type": "LoadImage", "inputs": {"image": ref_images[0] if len(ref_images) > 0 else "ref_image_1.png"}},
        "3": {"class_type": "LoadImage", "inputs": {"image": ref_images[1] if len(ref_images) > 1 else ref_images[0]}},
        "4": {"class_type": "HyperFlowSampler", "inputs": {"model": ["1", 0], "prompt": prompt_text, "image_1": ["2", 0], "image_2": ["3", 0], "seed": seed, "steps": 25, "cfg": 1.0, "shift": 5.0, "motion_bucket_id": 127}},
        "5": {"class_type": "SaveVideoWithAudio", "inputs": {"video": ["4", 0], "audio": ["4", 1], "filename_prefix": output_prefix, "fps": 24}}
    }

# Execution Results Registry
results = []

print("=" * 75, flush=True)
print(f">>> Executing Batch Queue: {len(BATCH_JOBS)} Jobs", flush=True)
print("=" * 75, flush=True)

for i, job in enumerate(BATCH_JOBS, 1):
    job_id = job.get("id", f"job_{i}")
    prefix = f"{i:02d}_{job_id}"
    print(f"\n[{i}/{len(BATCH_JOBS)}] Processing: {job_id}...", flush=True)
    t0 = time.time()
    
    # Fault tolerance: an error in one job NEVER breaks the rest of the batch
    try:
        wf = build_workflow(job["prompt"], job.get("ref_images", []), prefix, job.get("seed", 42))
        payload = json.dumps({"prompt": wf}).encode("utf-8")
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/prompt", data=payload, headers={"Content-Type": "application/json"})
        
        with urllib.request.urlopen(req) as resp:
            data = json.loads(resp.read().decode("utf-8"))
            prompt_id = data["prompt_id"]
        
        # Poll completion
        done = False
        while not done:
            time.sleep(4)
            with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/history/{prompt_id}") as h_resp:
                hist = json.loads(h_resp.read().decode("utf-8"))
                if prompt_id in hist:
                    status = hist[prompt_id].get("status", {})
                    if status.get("completed", False):
                        done = True
                    elif status.get("status_str") == "error":
                        raise RuntimeError(f"ComfyUI execution error: {status.get('messages')}")
        
        # Copy output immediately to /kaggle/working/outputs/ for instant streaming
        out_files = []
        comfy_out = "ComfyUI/output"
        if os.path.exists(comfy_out):
            for f in os.listdir(comfy_out):
                if f.startswith(prefix) and (f.endswith(".mp4") or f.endswith(".wav")):
                    src_p = os.path.join(comfy_out, f)
                    dst_p = os.path.join(OUTPUT_DIR, f)
                    shutil.copyfile(src_p, dst_p)
                    out_files.append(dst_p)
        
        elapsed = time.time() - t0
        print(f"✓ [{job_id}] Finished in {elapsed:.1f}s -> Saved: {out_files}", flush=True)
        results.append({"id": job_id, "status": "SUCCESS", "elapsed": f"{elapsed:.1f}s", "outputs": out_files})
        
    except Exception as e:
        elapsed = time.time() - t0
        print(f"❌ [{job_id}] Encountered Error: {e} (Continuing next job...)", flush=True)
        results.append({"id": job_id, "status": "FAILED", "elapsed": f"{elapsed:.1f}s", "error": str(e)})

print("\n" + "=" * 80, flush=True)
print("🏁 BATCH RUNNER FINISHED! FINAL SUMMARY:", flush=True)
print("=" * 80, flush=True)
for r in results:
    stat_icon = "✓" if r["status"] == "SUCCESS" else "❌"
    print(f" {stat_icon} {r['id']} ({r['status']} in {r['elapsed']}) -> {r.get('outputs', r.get('error'))}", flush=True)
print("=" * 80, flush=True)

# Strict Quota Auto-Termination: Kill server and exit immediately!
print("\n>>> Terminating server and exiting to preserve Kaggle GPU quota. Goodbye!", flush=True)
comfy_proc.terminate()
time.sleep(1)
sys.exit(0)
